# Voice2Reel — حلقة قياس وتحسين وإعادة تدريب (v2)

اختر **Runtime → Change runtime type → T4 GPU** وشغّل الخلايا بالترتيب.

هذا الدفتر **يقيس ولا يدّعي**:
1. بوابة أمامية على CPU: التشكيل الكامل، قراءة المصطلحات العربية/الإنجليزية، وراحة خطة الحركة (`examples/quality_v2/rounds.json`).
2. كاتب السكربت `Qwen2.5-1.5B` على GPU: `valid_rate` و`first_pass_rate` و`mean_content_f1` ودرجة مركّبة على **validation**.
3. ثلاث دورات تدريب موثّقة بزيادة عدد أمثلة التدريب/العصور؛ يُقيَّم **test مرة واحدة فقط** بعد اختيار أفضل دورة.

البيانات اصطناعية مؤلفة داخل المستودع (CC0)، بدون مراجعة خبراء، وليست دليلاً على دقة معلومات أو جودة صوت أو قبول بشري. درجة 90% المقيسة هنا تخص مهام البوابة الأمامية على تقسيم اختبار اصطناعي، **لا** جودة الفيديو النهائي. لا تكرّر ضبط الإعدادات على `test`.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile
from google.colab import files, drive

REPO = Path('/content/Voice2Reel')
if not (REPO / 'pyproject.toml').is_file():
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one repository ZIP from GitHub')
    destination = Path('/content/repo_upload')
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        if sum(info.file_size for info in archive.infolist()) > 50_000_000:
            raise ValueError('Expected source-only repository ZIP, under 50 MB')
        for info in archive.infolist():
            if not (destination / info.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(destination)
    projects = list(destination.rglob('pyproject.toml'))
    if len(projects) != 1:
        raise ValueError('Cannot identify repository root')
    shutil.copytree(projects[0].parent, REPO, dirs_exist_ok=True)
os.chdir(REPO)
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/voice2reel_quality_v2')
WORK.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO, '| Work:', WORK)


## تثبيت بيئة معزولة وGPU

نثبّت torch الخاص بـ CUDA ثم متطلبات النصوص/الاختبارات. لا نغيّر حزم Colab الأساسية.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.9.8'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '/content/v2r-quality-env'], check=True)
PY = '/content/v2r-quality-env/bin/python'

def pip_install(*args):
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PY, *args], check=True)

def run(*args):
    subprocess.run([PY, *map(str, args)], check=True)

pip_install('torch==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126')
pip_install('-r', 'requirements/text.txt', '-r', 'requirements/dev.txt')
pip_install('Pillow==13.0.0', 'arabic-reshaper==3.1.0')
run('-c', "import torch; assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU'; print(torch.cuda.get_device_name(0)); print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))")
freeze = subprocess.check_output([sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PY], text=True)
(WORK / 'environment.txt').write_text(freeze)


## فحص البيانات والقياس السابق على CPU

`scripts/check_data.py` يتحقق الآن من كل تقسيمات v2، وقابلية إعادة البناء بايت-ببايت، وعدم التسريب، وتطابقManifest. بوابة الجودة تُ fitted ثلاثة نماذج أمامية صغيرة فعلياً على CPU وتقيسها على validation ثم test مرة واحدة.


In [ ]:
run('scripts/check_data.py')
run('-m', 'voice2reel.quality_gate', '--output', WORK / 'quality_cpu.json')
gate = json.loads((WORK / 'quality_cpu.json').read_text())
print('CPU front-end rounds:', [r['score'] for r in gate['rounds']])
print('CPU front-end final test score:', gate['final_test_on_best_round']['score'])
run('-m', 'pytest', '-q', 'tests/test_arabic_text.py', 'tests/test_motion_plan.py',
    'tests/test_text_layout.py', 'tests/test_quality_gate.py', 'tests/test_evaluation.py')


## خط الأساس لكاتب السكربت

أول تشغيل ينزّل أوزان `Qwen/Qwen2.5-1.5B-Instruct` برقم المراجعة المثبّت. النتيجة على `validation` فقط.


In [ ]:
run('-m', 'voice2reel.evaluate', 'script', '--input', 'datasets/scripts/validation.jsonl',
    '--output', WORK / 'writer_baseline_validation.json')
baseline = json.loads((WORK / 'writer_baseline_validation.json').read_text())
print({k: baseline[k] for k in ('valid_rate', 'first_pass_rate', 'mean_content_f1', 'score')})


## دورات «تحسّن البيانات ← إعادة التدريب»

البيانات نفسها محسّنة ومولّدة داخل المستودع (`scripts/build_datasets.py`): 930/95/95 سكربت، 600 مثال تشكيل، 760 مثال مصطلحات، 1200 خطة حركة. الدورات التالية تغيّر **حجم بيانات التدريب وعدد العصور** بشكل موثّق، وتقيس كل دورة على نفس `validation`. توقف عند `TARGET` أو انتهاء الدورات؛ لا نلمس `test` هنا.


In [ ]:
TARGET = 0.90
CYCLES = [
    {'train_limit': 120, 'epochs': 1, 'note': 'small v2 sample'},
    {'train_limit': 360, 'epochs': 2, 'note': 'more v2 rows'},
    {'train_limit': 0, 'epochs': 3, 'note': 'all 930 v2 rows'},
]
history = []
for index, cycle in enumerate(CYCLES, 1):
    run_dir = WORK / f'writer_cycle_{index:02d}'
    command = ['-m', 'voice2reel.scriptwriter.train', '--output', run_dir,
               '--epochs', cycle['epochs']]
    if cycle['train_limit']:
        command += ['--train-limit', cycle['train_limit']]
    run(*command)
    evaluation = WORK / f'writer_cycle_{index:02d}_validation.json'
    run('-m', 'voice2reel.evaluate', 'script', '--input', 'datasets/scripts/validation.jsonl',
        '--adapter', run_dir / 'adapter', '--output', evaluation)
    report = json.loads(evaluation.read_text())
    training_metrics = json.loads((run_dir / 'metrics.json').read_text())
    row = {
        'cycle': index,
        'train_limit': cycle['train_limit'],
        'epochs': cycle['epochs'],
        'note': cycle['note'],
        'eval_loss': training_metrics.get('eval_loss'),
        'validation_score': report['score'],
        'valid_rate': report['valid_rate'],
        'first_pass_rate': report['first_pass_rate'],
        'mean_content_f1': report['mean_content_f1'],
        'adapter': str(run_dir / 'adapter'),
    }
    history.append(row)
    print(row)
    if row['validation_score'] >= TARGET:
        break
(WORK / 'cycles.json').write_text(json.dumps({
    'target': TARGET, 'baseline': baseline['score'], 'cycles': history,
    'note': 'validation only; test is evaluated once after selecting the best cycle',
}, ensure_ascii=False, indent=2))


## التقييم النهائي مرة واحدة

نختار أفضل دورة حسب `validation_score`، ثم نقيس baseline وأفضل adapter على `test` فقط. إن لم تصل الدرجة إلى الهدف، يُكتب ذلك بصراحة في التقرير بدل ادّعاء 90%.


In [ ]:
saved = json.loads((WORK / 'cycles.json').read_text())
history = saved['cycles']
best = max(history, key=lambda row: row['validation_score'])
for adapter, label in ((None, 'baseline'), (Path(best['adapter']), 'best_finetuned')):
    command = ['-m', 'voice2reel.evaluate', 'script', '--input', 'datasets/scripts/test.jsonl',
               '--output', WORK / f'writer_{label}_test.json']
    if adapter:
        command += ['--adapter', adapter]
    run(*command)
final_report = {
    'best_cycle': best,
    'target': TARGET,
    'baseline_validation': saved['baseline'],
    'baseline_test': json.loads((WORK / 'writer_baseline_test.json').read_text())['score'],
    'finetuned_test': json.loads((WORK / 'writer_best_finetuned_test.json').read_text())['score'],
    'cpu_front_end_test': gate['final_test_on_best_round']['score'],
    'reached_target': json.loads((WORK / 'writer_best_finetuned_test.json').read_text())['score'] >= TARGET,
    'warning': 'Synthetic authored splits; schema/word-overlap metrics do not prove factual accuracy, voice naturalness, or human acceptance.',
}
(WORK / 'final_report.json').write_text(json.dumps(final_report, ensure_ascii=False, indent=2))
print(json.dumps(final_report, ensure_ascii=False, indent=2))


## حفظ النتائج

نزّل تقرير القياس وأفضل adapter. الـ adapter يحتاج أوزان النموذج الأساسي عند التشغيل، ولا يحسّن تلقائياً كاتب الإنتاج `Qwen3-4B/GGUF`.


In [ ]:
import shutil
archive = shutil.make_archive('/content/voice2reel_quality_v2', 'zip', WORK)
files.download(archive)
print('Downloaded:', archive)
